In [13]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# --- CONFIGURATION ---
# These parameters MUST match the ones used in the training/generation scripts.
SEQ_LEN = 12      # Input history length
PRED_LEN = 6  # Output forecast horizon
DATA_PATH = '../../../ICL4DT/data/time_series_datasets/ETTm2.csv'
HTI_DATA_DIR = 'hti_data_long'

# The quantiles of the expert models you want to visualize
QUANTILES_TO_LOAD = [0.01, 0.1, 0.25, 0.5, 0.75,0.9, 0.99]

print("Loading original dataset...")
df = pd.read_csv(DATA_PATH)
data = df['OT'].values.astype(float)

print(f"Full dataset shape: {data.shape}")

# Recreate the exact train/val/test split to fit the scaler correctly
train_split_idx = int(len(data) * 0.7)
val_split_idx = int(len(data) * 0.98)

# Isolate the original, unscaled test data for ground truth comparison
original_test_data = data[val_split_idx:]

# Fit the scaler ONLY on the training data to prevent data leakage
print("Fitting MinMaxScaler on the training data portion...")
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(data[:train_split_idx].reshape(-1, 1))

hti_datasets = {}
all_forecasts_unscaled = {}

print("Loading HTI datasets and unscaling forecasts...")

for q in QUANTILES_TO_LOAD:
    # Construct filename (e.g., hti_data_q05.pt)
    filename = f"hti_data_q{str(q).replace('.', '')}.pt"
    
    try:
        # Load the entire [history, forecast] tensor
        hti_datasets[q] = torch.load(filename)
        data = hti_datasets[q]
        print(f" -> Loaded '{filename}' with shape: {hti_datasets[q].shape}")
        data_unscaled = scaler.inverse_transform(data)
        data_unscaled = torch.tensor(data_unscaled, dtype=torch.float32)
        all_forecasts_unscaled[q] = data_unscaled
        
    except FileNotFoundError:
        print(f" -> WARNING: Could not find file {filename}. Skipping.")

print("\nForecasts are now unscaled and ready for plotting.")


Loading original dataset...
Full dataset shape: (69680,)
Fitting MinMaxScaler on the training data portion...
Loading HTI datasets and unscaling forecasts...
 -> Loaded 'hti_data_q001.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q01.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q025.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q05.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q075.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q09.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q099.pt' with shape: torch.Size([1377, 18])

Forecasts are now unscaled and ready for plotting.


In [14]:
all_forecasts_unscaled

{0.01: tensor([[34.1700, 34.3895, 35.0485,  ..., 39.3967, 39.3639, 38.6305],
         [34.3895, 35.0485, 35.4885,  ..., 40.4911, 40.4816, 39.6985],
         [35.0485, 35.4885, 36.1475,  ..., 41.3067, 41.3188, 40.4904],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 43.0934, 43.2283, 42.1122],
         [47.7445, 47.9640, 48.1835,  ..., 42.6514, 42.7661, 41.6710],
         [47.9640, 48.1835, 48.1835,  ..., 42.8004, 42.9307, 41.8653]]),
 0.1: tensor([[34.1700, 34.3895, 35.0485,  ..., 41.6403, 41.5417, 41.6055],
         [34.3895, 35.0485, 35.4885,  ..., 43.0979, 43.0609, 43.1884],
         [35.0485, 35.4885, 36.1475,  ..., 44.0633, 44.0316, 44.1764],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 45.1481, 44.6594, 44.2286],
         [47.7445, 47.9640, 48.1835,  ..., 44.8140, 44.3484, 43.9405],
         [47.9640, 48.1835, 48.1835,  ..., 45.2202, 44.8237, 44.4806]]),
 0.25: tensor([[34.1700, 34.3895, 35.0485,  ..., 42.4610, 42.6745, 42.7477],
         [34.3895, 35.0485, 

In [15]:
combined = torch.stack([all_forecasts_unscaled[q] for q in QUANTILES_TO_LOAD], dim=0)

torch.save(combined, 'hti_data_combined.pt')

In [16]:
combined.shape

torch.Size([7, 1377, 18])

In [17]:
combined

tensor([[[34.1700, 34.3895, 35.0485,  ..., 39.3967, 39.3639, 38.6305],
         [34.3895, 35.0485, 35.4885,  ..., 40.4911, 40.4816, 39.6985],
         [35.0485, 35.4885, 36.1475,  ..., 41.3067, 41.3188, 40.4904],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 43.0934, 43.2283, 42.1122],
         [47.7445, 47.9640, 48.1835,  ..., 42.6514, 42.7661, 41.6710],
         [47.9640, 48.1835, 48.1835,  ..., 42.8004, 42.9307, 41.8653]],

        [[34.1700, 34.3895, 35.0485,  ..., 41.6403, 41.5417, 41.6055],
         [34.3895, 35.0485, 35.4885,  ..., 43.0979, 43.0609, 43.1884],
         [35.0485, 35.4885, 36.1475,  ..., 44.0633, 44.0316, 44.1764],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 45.1481, 44.6594, 44.2286],
         [47.7445, 47.9640, 48.1835,  ..., 44.8140, 44.3484, 43.9405],
         [47.9640, 48.1835, 48.1835,  ..., 45.2202, 44.8237, 44.4806]],

        [[34.1700, 34.3895, 35.0485,  ..., 42.4610, 42.6745, 42.7477],
         [34.3895, 35.0485, 35.4885,  ..., 43

In [18]:
combined_min = combined.min().item()
combined_max = combined.max().item()
print(f"Min value in combined: {combined_min}")
print(f"Max value in combined: {combined_max}")

Min value in combined: 24.570924758911133
Max value in combined: 57.138267517089844
